In [19]:
# tensorflow pytorch를 중복으로 사용시 OpenMP 라이브러리 실행 허용
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

In [20]:
from transformers import AutoTokenizer, AutoModel
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [21]:
text = "이 영화는 정말 재미있다."

# 1. 토크나이저
inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
inputs

# 2. 토큰된 단어 확인
tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# [CLS] => 문장전체 정보를 가지고 있음
# ## 서브워드
# [SEP] => 문장끝

['[CLS]', '이', '영화', '##는', '정말', '재미있', '##다', '.', '[SEP]']

In [22]:
# 임베딩 적용 (768개)
model_name = "klue/bert-base"
model = AutoModel.from_pretrained(model_name)

output = model(
    input_ids = inputs['input_ids'],
    attention_mask = inputs['attention_mask']
)

# [1, 9, 768] => [n, 토큰개수, 768]
print(output['last_hidden_state'].shape)
# CLS 벡터
print(output['last_hidden_state'][:, 0])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 9, 768])
tensor([[ 5.6780e-01, -1.0617e+00,  2.3131e-02,  3.4950e-01,  1.2665e+00,
         -7.2037e-02,  1.0242e-02, -5.0413e-01,  1.4125e+00, -1.6481e-01,
         -2.5494e-01,  5.9440e-01,  1.0956e+00, -4.7720e-01,  5.0787e-01,
          1.4413e-01, -7.8498e-01,  1.1544e+00, -8.6248e-02,  1.9296e-02,
         -7.3924e-01, -5.3338e-01,  9.4760e-01, -3.6030e-01,  6.1731e-02,
         -7.4392e-01, -7.0290e-02,  1.3067e-01, -1.2668e-01,  9.8234e-02,
         -5.8464e-01,  1.6957e-01,  4.5356e-01,  5.6845e-02,  4.3755e-01,
          9.3975e-01,  8.2929e-01,  1.0157e+00, -9.4954e-02, -8.9043e-01,
          2.9437e-01,  5.2462e-01,  9.8190e-01,  3.0289e-01,  3.4851e-01,
          6.1853e-02,  6.8814e-01, -1.0501e+00, -3.7598e-01, -1.6054e-01,
         -6.3847e-01,  5.1376e-01, -3.1538e-01,  2.6797e-01,  1.0478e+00,
          1.5806e-01, -1.0376e+00, -3.8408e-01,  1.7733e-01,  1.2582e-01,
         -9.7111e-01, -1.6613e-01,  4.2502e-01,  3.9352e-02,  9.6629e-01,
         -8.48

In [23]:
import pandas as pd

df = pd.read_csv("http://114.207.245.181:13000/csv/sentiment_data.csv")
df

,document,label
0,정말 재미있는 영화였다,1
1,최고의 경험이었다,1
2,너무 만족한다,1
3,추천하고 싶다,1
4,정말 훌륭하다,1
5,최악의 영화였다,0
6,돈이 아깝다,0
7,너무 실망했다,0
8,다시는 안 본다,0
9,형편없다,0


In [24]:
texts = df['document'].tolist()
labels = df['label'].tolist()

texts, labels

(['정말 재미있는 영화였다',
  '최고의 경험이었다',
  '너무 만족한다',
  '추천하고 싶다',
  '정말 훌륭하다',
  '최악의 영화였다',
  '돈이 아깝다',
  '너무 실망했다',
  '다시는 안 본다',
  '형편없다'],
 [1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

In [25]:
# 1. 토크나이저
from transformers import AutoTokenizer, AutoModel

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
    max_length=100
)
inputs

{'input_ids': tensor([[    2,  3944,  6001,  2259,  3771,  2507,  2062,     3],
        [    2,  3841,  2079,  4043,  2052,  2359,  2062,     3],
        [    2,  3760,  4671,  4538,     3,     0,     0,     0],
        [    2,  4635, 19521,  1335,  2062,     3,     0,     0],
        [    2,  3944,  5825,  2205,  2062,     3,     0,     0],
        [    2,  7966,  2079,  3771,  2507,  2062,     3,     0],
        [    2,   850,  2052, 15021,  2062,     3,     0,     0],
        [    2,  3760,  7334,  2371,  2062,     3,     0,     0],
        [    2,  3690,  2259,  1378,  4919,     3,     0,     0],
        [    2, 22314,  2062,     3,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],


In [26]:
# 2. Dataset 생성
import torch
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(
    inputs['input_ids'],
    inputs['attention_mask'],
    torch.tensor(labels, dtype=torch.float32)
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [34]:
# 3. 모델생성
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

class BertClassifier(nn.Module):

    # 레이어 생성
    def __init__(self):
        super().__init__()

        #=================================================
        # bert 모델
        self.bert = AutoModel.from_pretrained(model_name)

        # 빠른 테스트 bert 고정
        for param in self.bert.parameters():
            param.requires_grad = False

        # bert 모델 결과 hidden_size추출 (768)
        hidden_size = self.bert.config.hidden_size
        #=================================================
        
        # 과적합 방지용 랜덤 20%는 제거
        self.dropout = nn.Dropout(0.2)

        # 회귀 768 => 128로 변환
        self.fc1 = nn.Linear(hidden_size, 128)

        # activation => relu
        self.relu = nn.ReLU()

        # 회귀 128 => 1로 변환
        self.fc2 = nn.Linear(128, 1)

    # 실행
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask
        )

        cls = outputs['last_hidden_state'][:, 0]
        x = self.dropout(cls)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

In [41]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = BertClassifier().to(device)

# binaray_crossentropy
criterion = nn.BCEWithLogitsLoss()

# L2 규제
# filter 를 통해서 parameters 중에서 requires_grad =False는 제거
# learning_rate는 0.001
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr = 0.001
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad() # 기울기 초기화
        logits = model(input_ids = input_ids, attention_mask = attention_mask) # 모델 학습
        loss = criterion(logits.squeeze(), labels) # 로그 계산
        loss.backward() # 기울기 계산
        optimizer.step() # 가중치 업데이트
        total_loss += loss.item() # 전체 loss값 누적

    print(f"Epoch {epoch+1} Loss {total_loss/len(loader):.4f}") # epoch와 loss값 출력

Epoch 1 Loss 0.8790
Epoch 2 Loss 0.6029
Epoch 3 Loss 0.4764
Epoch 4 Loss 0.4152
Epoch 5 Loss 0.3390
Epoch 6 Loss 0.2352
Epoch 7 Loss 0.1833
Epoch 8 Loss 0.1326
Epoch 9 Loss 0.1140
Epoch 10 Loss 0.0803
